# Lektion 09 - Metakognition Designmönster


## Konfiguration

Detta anteckningsblock demonstrerar designmönstret för metakognition med Microsoft Agent Framework.

**Förutsättningar:**
- Azure OpenAI-distribution konfigurerad via miljövariabler
- Azure CLI autentiserad (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Vad är metakognition?

Metakognition är **tänkande om tänkande**. I samband med AI-agenter innebär det att bygga agenter som kan:

- **Självreflektera** över sina egna utdata och sin resoneringsprocess
- **Upptäcka fel** och återhämta sig smidigt istället för att misslyckas tyst
- **Utvärdera** om deras svar är fullständiga och hjälpsamma
- **Anpassa** sin strategi när en initial metod inte fungerar (t.ex. att falla tillbaka på ett backupsystem)

En metakognitiv agent svarar inte bara på frågor — den övervakar sin egen prestation och justerar sig i farten.


## Primära och reservverktyg

Ett vanligt metakognitionsmönster är **fallback-strategin**. Agenten försöker ett primärt verktyg först; om det misslyckas (t.ex. ett 404-fel) känner agenten igen felet och byter transparent till ett reservverktyg.

Detta återspeglar verkliga system där primära tjänster kan vara otillgängliga och agenter måste självdiagnostisera problemet innan de väljer en alternativ väg.

Nedan definierar vi två verktyg för flygsökning:
- **Primary** — täcker Paris, Tokyo och Barcelona
- **Backup** — täcker Berlin, Sydney och New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Självreflekterande agent med felåterställning

Agenten nedan är instruerad att först försöka använda det primära flygsystemet, känna igen fel och på ett transparent sätt falla tillbaka till reservsystemet. Efter varje svar utvärderar den kort om den fullständigt besvarade användarens fråga.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Självutvärderingsmönster

En annan aspekt av metakognition är **självutvärdering**: en separat agent (eller samma agent i en andra genomgång) granskar ett svar för fullständighet, noggrannhet och hjälpsamhet.

Nedan skapar vi en `ResponseEvaluator`-agent som poängsätter reseagenters svar utifrån tre dimensioner.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Sammanfattning

I den här lektionen lärde du dig hur du bygger **metakognitiva agenter** med Microsoft Agent Framework:

- **Självreflektion**: Agenter som övervakar sitt eget resonemang och på ett transparent sätt kommunicerar vad som har hänt.
- **Återhämtning vid fel med fallback**: Ett mönster med primärt + backup-verktyg där agenten upptäcker fel (t.ex. 404-fel) och automatiskt försöker en alternativ källa.
- **Självutvärdering**: En separat utvärderingsagent som poängsätter svar för fullständighet, noggrannhet och hjälpsamhet.

Dessa mönster gör agenter mer robusta, transparenta och pålitliga — avgörande egenskaper för driftsättning i produktionsmiljöer.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Ansvarsfriskrivning:
Detta dokument har översatts med hjälp av AI-översättningstjänsten Co-op Translator (https://github.com/Azure/co-op-translator). Vi strävar efter noggrannhet, men var medveten om att automatiska översättningar kan innehålla fel eller felaktigheter. Originaldokumentet på sitt ursprungsspråk ska betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
